In [11]:
import chromadb
from sentence_transformers import SentenceTransformer


# =========================================================
# 1. 기본 설정
# =========================================================

EMBEDDING_MODEL_NAME = "jhgan/ko-sroberta-multitask"

# ChromaDB를 만들 때 사용한 실제 경로와 동일하게 설정
#CHROMA_PATH = "../chroma_db"
CHROMA_PATH = "../../chroma_db"
# ChromaDB를 만들 때 사용한 collection 이름
COLLECTION_NAME = "maplestory_guides"


# =========================================================
# 2. 임베딩 모델 / ChromaDB 연결
# =========================================================

embed_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

collection = client.get_collection(
    name=COLLECTION_NAME
)

print("Collection:", collection.name)
print("문서 수:", collection.count())


# =========================================================
# 3. 질문 분류를 위한 키워드
# =========================================================

JOB_KEYWORDS = [
    # 직업 관련 일반 단어
    "직업",
    "직업추천",
    "직업 추천",
    "주스탯",
    "주 스탯",
    "주스텟",
    "무기",
    "전직",

    # 직업군
    "전사",
    "마법사",
    "궁수",
    "도적",
    "해적",

    # 예시 직업명
    "히어로",
    "팔라딘",
    "다크나이트",
    "아크메이지",
    "비숍",
    "보우마스터",
    "신궁",
    "패스파인더",
    "나이트로드",
    "섀도어",
    "듀얼블레이드",
    "바이퍼",
    "캡틴",
    "캐논슈터",

    "소울마스터",
    "미하일",
    "플레임위자드",
    "윈드브레이커",
    "나이트워커",
    "스트라이커",

    "아란",
    "에반",
    "메르세데스",
    "팬텀",
    "루미너스",
    "은월",

    "데몬슬레이어",
    "데몬어벤져",
    "블래스터",
    "배틀메이지",
    "와일드헌터",
    "메카닉",
    "제논",

    "카이저",
    "카인",
    "카데나",
    "엔젤릭버스터",

    "아델",
    "일리움",
    "아크",
    "칼리",

    "호영",
    "라라",

    "제로",
    "키네시스",
    "렌",
]


ITEM_KEYWORDS = [
    "확률",
    "아이템",
    "장비",
    "큐브",
    "잠재",
    "잠재능력",
    "에디셔널",
    "스타포스",
    "강화",
    "획득 확률",
]


# =========================================================
# 4. 질문에 따라 검색 범위 결정
# =========================================================

def get_search_filter(query):
    """
    사용자 질문을 간단한 키워드 기반으로 분류하여
    검색할 source를 결정합니다.

    job 질문  -> {"source": "jobs"}
    item 질문 -> {"source": "items"}
    그 외     -> None (전체 문서 검색)
    """

    query = query.strip().lower()

    # 직업 관련 질문
    if any(keyword.lower() in query for keyword in JOB_KEYWORDS):
        return {"source": "jobs"}

    # 아이템 / 확률 관련 질문
    if any(keyword.lower() in query for keyword in ITEM_KEYWORDS):
        return {"source": "items"}

    # 분류가 확실하지 않을 경우 전체 검색
    return None


# =========================================================
# 5. Top-K Retriever
# =========================================================

def retrieve_top_k(
    query,
    collection,
    embed_model,
    top_k=8,
    where=None,
):
    query = query.strip()

    if not query:
        raise ValueError("검색 질문이 비어 있습니다.")

    if top_k < 1:
        raise ValueError("top_k는 1 이상이어야 합니다.")

    document_count = collection.count()

    if document_count == 0:
        return []

    # DB 문서와 동일한 임베딩 모델 사용
    query_embedding = embed_model.encode(
        [query],
        normalize_embeddings=True,
)

    if query_embedding.ndim != 2:
        raise ValueError(
            f"질문 임베딩 차원 오류: {query_embedding.shape}"
        )

    if query_embedding.shape[0] != 1:
        raise ValueError(
            f"질문은 1개여야 합니다: {query_embedding.shape}"
        )

    query_kwargs = {
        "query_embeddings": query_embedding.tolist(),
        "n_results": min(top_k, document_count),
        "include": [
            "documents",
            "metadatas",
            "distances",
        ],
    }

    # source filtering
    if where:
        query_kwargs["where"] = where

    raw_results = collection.query(**query_kwargs)

    results = []

    for index, chunk_id in enumerate(
        raw_results["ids"][0]
):
        distance = raw_results["distances"][0][index]

        results.append(
            {
                "rank": index + 1,
                "id": chunk_id,
                "page_content": (
                    raw_results["documents"][0][index]
                ),
                "metadata": (
                    raw_results["metadatas"][0][index]
                ),
                "distance": distance,
                "score": 1.0 - distance,
            }
        )

    return results


# =========================================================
# 6. Router + Retriever
# =========================================================

def search_documents(
    query,
    collection,
    embed_model,
    top_k=5,
):
    """
    질문을 분류한 후 적절한 source에서 Top-K 검색
    """

    where = get_search_filter(query)

    print("=" * 80)
    print(f"질문: {query}")

    if where is None:
        print("검색 범위: 전체 문서")
    else:
        print(f"검색 범위: {where['source']}")

    print("=" * 80)

    results = retrieve_top_k(
        query=query,
        collection=collection,
        embed_model=embed_model,
        top_k=top_k,
        where=where,
)

    return results


# =========================================================
# 7. 검색 테스트
# =========================================================

query = "표창을 사용하는 직업 모두 정리해봐"

results = search_documents(
    query=query,
    collection=collection,
    embed_model=embed_model,
    top_k=3,
)


# =========================================================
# 8. 결과 출력
# =========================================================

for result in results:
    print("=" * 80)

    print(f"순위: {result['rank']}")
    print(f"점수: {result['score']:.4f}")

    print(
        "source:",
        result["metadata"].get("source")
    )

    print(
        "문서명:",
        result["metadata"].get("name")
    )

    print(
        "URL:",
        result["metadata"].get("url")
    )

    print()

    print(result["page_content"])

    print()
    

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2632.85it/s]


Collection: maplestory_guides
문서 수: 3694
질문: 표창을 사용하는 직업 모두 정리해봐
검색 범위: jobs
순위: 1
점수: 0.4719
source: jobs
문서명: 나이트로드
URL: https://maplestory.nexon.com/Guide/N23Job/View/30

직업명: 나이트로드
별명: 그림자 속에 숨은 존재
설명: 다양한 종류의 표창을 능수능란하게 다루는 도적입니다. 끊임없는 표창 투척과 함께 암살자의 표식, 부적, 두루마리로 더욱 많은 표창을 소환하여 적에게 강력한 피해를 주고 전장을 제압합니다.
주스탯: LUK (행운)
무기: 표창, 아대

순위: 2
점수: 0.4298
source: jobs
문서명: 보우마스터
URL: https://maplestory.nexon.com/Guide/N23Job/View/23

직업명: 보우마스터
별명: 속사의 정점
설명: 활의 정점에 도달해 다양한 화살로 적을 섬멸하는 궁수입니다. 화살을 연속적으로 발사하는 속사 공격과 상황에 맞춰 기능을 선택할 수 있는 추가 화살을 끊임없이 퍼부어 전장을 뒤덮습니다.
주스탯: DEX (민첩)
무기: 활

순위: 3
점수: 0.3988
source: jobs
문서명: 배틀메이지
URL: https://maplestory.nexon.com/Guide/N23Job/View/17

직업명: 배틀메이지
별명: 최전선의 마법사
설명: 실전에 특화되어 근접전이 가능한 마법사입니다. 높은 기동력으로 접근해 스태프를 휘둘러 공격하며, 어둠의 힘으로 적을 응징하고 다양한 오라로 동료를 지원합니다.
주스탯: INT (지력)
무기: 스태프



In [19]:
def build_context(results):
    context_parts = []

    for result in results:
        metadata = result.get("metadata") or {}

        context_parts.append(
            f"""
[검색 결과 {result.get("rank")}]
문서 유형: {metadata.get("source", "unknown")}
문서 제목: {metadata.get("name", "제목 없음")}
문서 ID: {result.get("id", "")}
유사도 점수: {result.get("score", 0):.4f}

내용:
{result.get("page_content", "")}

출처 URL:
{metadata.get("url", "")}
""".strip()
        )

    return "\n\n---\n\n".join(context_parts)


In [20]:
#test
test_context = build_context(results)

print(test_context)
print(type(test_context))
print("히어로" in test_context)
print("문서 유형:" in test_context)
print("출처 URL:" in test_context)

[검색 결과 1]
문서 유형: jobs
문서 제목: 나이트로드
문서 ID: chunk_1472
유사도 점수: 0.4719

내용:
직업명: 나이트로드
별명: 그림자 속에 숨은 존재
설명: 다양한 종류의 표창을 능수능란하게 다루는 도적입니다. 끊임없는 표창 투척과 함께 암살자의 표식, 부적, 두루마리로 더욱 많은 표창을 소환하여 적에게 강력한 피해를 주고 전장을 제압합니다.
주스탯: LUK (행운)
무기: 표창, 아대

출처 URL:
https://maplestory.nexon.com/Guide/N23Job/View/30

---

[검색 결과 2]
문서 유형: jobs
문서 제목: 보우마스터
문서 ID: chunk_1465
유사도 점수: 0.4298

내용:
직업명: 보우마스터
별명: 속사의 정점
설명: 활의 정점에 도달해 다양한 화살로 적을 섬멸하는 궁수입니다. 화살을 연속적으로 발사하는 속사 공격과 상황에 맞춰 기능을 선택할 수 있는 추가 화살을 끊임없이 퍼부어 전장을 뒤덮습니다.
주스탯: DEX (민첩)
무기: 활

출처 URL:
https://maplestory.nexon.com/Guide/N23Job/View/23

---

[검색 결과 3]
문서 유형: jobs
문서 제목: 배틀메이지
문서 ID: chunk_1458
유사도 점수: 0.3988

내용:
직업명: 배틀메이지
별명: 최전선의 마법사
설명: 실전에 특화되어 근접전이 가능한 마법사입니다. 높은 기동력으로 접근해 스태프를 휘둘러 공격하며, 어둠의 힘으로 적을 응징하고 다양한 오라로 동료를 지원합니다.
주스탯: INT (지력)
무기: 스태프

출처 URL:
https://maplestory.nexon.com/Guide/N23Job/View/17
<class 'str'>
False
True
True


In [21]:
assert build_context([]) == ""

In [22]:
question = "히어로가 사용하는 무기는 뭐야?"

prompt_text = f"""
당신은 메이플스토리 정보 안내 챗봇입니다.

반드시 아래 Context에 있는 정보만 사용해서 답변하세요.
Context에 답이 없으면 모른다고 답변하세요.

[Context]
{test_context}

[Question]
{question}
"""

print(prompt_text)
assert test_context in prompt_text
assert question in prompt_text


당신은 메이플스토리 정보 안내 챗봇입니다.

반드시 아래 Context에 있는 정보만 사용해서 답변하세요.
Context에 답이 없으면 모른다고 답변하세요.

[Context]
[검색 결과 1]
문서 유형: jobs
문서 제목: 나이트로드
문서 ID: chunk_1472
유사도 점수: 0.4719

내용:
직업명: 나이트로드
별명: 그림자 속에 숨은 존재
설명: 다양한 종류의 표창을 능수능란하게 다루는 도적입니다. 끊임없는 표창 투척과 함께 암살자의 표식, 부적, 두루마리로 더욱 많은 표창을 소환하여 적에게 강력한 피해를 주고 전장을 제압합니다.
주스탯: LUK (행운)
무기: 표창, 아대

출처 URL:
https://maplestory.nexon.com/Guide/N23Job/View/30

---

[검색 결과 2]
문서 유형: jobs
문서 제목: 보우마스터
문서 ID: chunk_1465
유사도 점수: 0.4298

내용:
직업명: 보우마스터
별명: 속사의 정점
설명: 활의 정점에 도달해 다양한 화살로 적을 섬멸하는 궁수입니다. 화살을 연속적으로 발사하는 속사 공격과 상황에 맞춰 기능을 선택할 수 있는 추가 화살을 끊임없이 퍼부어 전장을 뒤덮습니다.
주스탯: DEX (민첩)
무기: 활

출처 URL:
https://maplestory.nexon.com/Guide/N23Job/View/23

---

[검색 결과 3]
문서 유형: jobs
문서 제목: 배틀메이지
문서 ID: chunk_1458
유사도 점수: 0.3988

내용:
직업명: 배틀메이지
별명: 최전선의 마법사
설명: 실전에 특화되어 근접전이 가능한 마법사입니다. 높은 기동력으로 접근해 스태프를 휘둘러 공격하며, 어둠의 힘으로 적을 응징하고 다양한 오라로 동료를 지원합니다.
주스탯: INT (지력)
무기: 스태프

출처 URL:
https://maplestory.nexon.com/Guide/N23Job/View/17

[Question]
히어로가 사용하는 무기는 뭐야?



In [31]:

import os
from getpass import getpass
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

In [32]:
def extract_answer(response):
    if isinstance(response, str):
        return response.strip()

    if hasattr(response, "content"):
        return str(response.content).strip()

    return str(response).strip()

In [33]:
assert extract_answer("히어로는 검과 도끼를 사용합니다.") == (
    "히어로는 검과 도끼를 사용합니다."
)

In [34]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
당신은 메이플스토리 정보 안내 챗봇입니다.

반드시 제공된 Context를 기반으로 답변하세요.
Context에 없는 정보는 임의로 만들어내지 마세요.
답변은 간결하고 정확하게 작성하세요.
""".strip(),
        ),
        (
            "human",
            """
[Context]
{context}

[Question]
{question}
""".strip(),
        ),
    ]
)

In [35]:
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough,
)

retrieve_runnable = RunnableLambda(
    lambda user_question: search_documents(
        query=user_question,
        collection=collection,
        embed_model=embed_model,
        top_k=5,
    )
)

rag_chain = (
    {
        "context": retrieve_runnable | RunnableLambda(build_context),
        "question": RunnablePassthrough(),
    }
    | prompt
    | model
    | StrOutputParser()
)